# Day 30：完整RAG系统搭建(核心日)：端到端全链路、引用溯源与防幻觉兜底（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 5 周**：第5周：企业级 RAG 全链路、高级检索优化与评估  
> **GitLab 工单**：`STARMIND-030` ｜ **开发分支**：`feat/day30-end-to-end-rag-system`  
> **配套 HTML5 课件**：`courseware/day30/index.html` ｜ **独立版本快照**：`versions/day30/`

---

## 🎙️ 老陈开场旁白

各位早！今天是整个第三阶段最重要的「核心日」——Day 30！前几天大家一直在打磨零部件，今天咱们把「文档加载 -> 中文切分 -> 向量化入库 -> 相似度检索 -> 强约束 Prompt 增强生成 -> 引用来源标注 -> 兜底拒答」一整条企业级 RAG 流水线端到端打通！老陈特别强调两个决定企业系统能不能上线的生死线：**第一是引用溯源（Citation）**，员工看到回答必须能一眼核对出自《哪份文档》的哪一段；**第二是「我不知道」防幻觉兜底**，当知识库里根本没有相关资料（或者最高检索相似度低于阈值）时，系统必须果断拒答，宁可说「知识库暂无记录，请联系 HR/IT」，也绝不允许大模型胡编乱造一条假报销政策！

### 承上启下
- **昨日基线**：过去 5 天，我们已经像造精密手表一样，把 ChatModel、LCEL 管道、多格式 Loader、中文递归 Splitter 和持久化向量数据库全部亲手打造完毕。今天，是把所有齿轮咬合在一起、让第一版「星辰企业知识库问答系统」真正转起来的核心里程碑日！
- **明日预告**：明天（Day 31）我们将迎来第 5 周的实战周测与 RAG 效果调优实验日，用量化指标（Hit@K / MRR）系统对比不同分块大小、Top-K 和 Embedding 方案的效果差异。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day30'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-030）

杭州星辰数科新员工入职首月平均向 HR、财务、IT 咨询 15 个重复性制度问题。公司要求上线第一版内部知识库命令行/API 问答原型（Naive RAG v1.0），实现 24 小时自助精准答疑。

**今日迭代目标：**
- 串联 Day 28~29 组件，实现完整的离线索引构建（Indexing）与在线检索生成（Retrieval & Generation）双流水线。
- 设计强约束 RAG System Prompt，强制模型仅依据检索到的 `[来源1]`、`[来源2]` 作答并在答案末尾附带结构化引用列表。
- 实现基于相似度阈值（`score_threshold`）的「我不知道」兜底拒答机制，并交付星辰知识库 CLI 问答工具。

**验收标准：**
- 针对星辰知识库内制度问题（报销/年假/代码发布/VPN），能精准给出答案并标注正确 `[来源N]` 文档名与章节。
- 针对库外无关问题（如「太阳系有几颗行星」或「星辰股票代码是多少」），触发低分阈值拒答，绝不编造。
- 支持按部门 `dept` 过滤查询，命令行与 pytest 单元测试 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. RAG Prompt 编写的四大黄金法则

一个工业级 RAG System Prompt 必须包含以下四要素：
1. **明确知识边界**：「你只能根据下方 `<context>` 提供的参考资料回答问题，严禁使用你的预训练外部知识补充公司制度。」
2. **强制引用标注**：「在每个关键结论后使用 `[来源1]`、`[来源2]` 标明出处。」
3. **明确拒答话术**：「如果参考资料中没有足够信息回答该问题，请直接回答：『抱歉，星辰知识库中暂未找到与该问题相关的确切依据，建议联系对应部门人工确认。』」
4. **防提示词注入隔离**：把检索回来的文档用 `<context>...</context>` 标签包起来，提醒模型里面是参考数据而非系统指令。

> **⚠️ 老陈避坑**：踩坑提醒：RAG 生成环节的 `temperature` 一定要设为 `0.0 ~ 0.1`！知识库问答要的是精准忠实，不需要大模型发散创造力。

### 2. 2. 生产环境 LangChain LCEL RAG 真实写法对照

在生产环境用 LangChain LCEL 组装带引用来源的 RAG 链，标准范式如下：
```python
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_type='similarity_score_threshold', search_kwargs={'k': 4, 'score_threshold': 0.5})
rag_chain = (
    RunnableParallel(docs=retriever, question=RunnablePassthrough())
    | RunnablePassthrough.assign(context=lambda x: format_docs_with_citations(x['docs']))
    | RunnablePassthrough.assign(answer=rag_prompt | llm | StrOutputParser())
)
```

> **⚠️ 老陈避坑**：踩坑提醒：不要直接用 `RetrievalQA`（已被官方标记 Deprecated），用 LCEL 可以轻松同时返回 `answer` 和 `docs` 原始切片给前端渲染引用卡片。

### 3. 3. 为什么「拒答阈值」不能拍脑袋定死？

不同 Embedding 模型的相似度分数分布完全不同！比如 `bge-large-zh-v1.5` 的无关文本余弦相似度也经常在 `0.35~0.45` 之间（各向异性现象），而哈希/稀疏特征的分数可能集中在 `0.15~0.70`。正确做法是用 20 条「库内正样本」+ 20 条「库外负样本」画一张分数分布直方图，在精确率和召回率之间选出最优阈值。

> **⚠️ 老陈避坑**：踩坑提醒：阈值设太高会误杀正常提问（过度拒答，员工骂系统傻），设太低会放进无关噪音引发幻觉。明天（Day 31）我们会用自动化实验精确标定它！


## 💻 实操步骤 1：步骤一：组装端到端 EnterpriseRAGPipeline（含引用溯源与兜底拒答）

- **目标文件**：`versions/day30/src/day30_rag_pipeline.py`
- **老陈代码旁白**：咱们把前面几天的组件正式合体！`EnterpriseRAGPipeline` 封装了文档入库、相似度阈值门控、带 `[来源N]` 编号的上下文构建、忠实生成以及结构化 `RAGResponse` 返回。


In [ ]:
"""
Day 30 - 步骤一：端到端企业级 RAG 核心流水线（检索 + 引用溯源 + 防幻觉兜底）
对应生产栈：LangChain LCEL + Chroma/Milvus + DeepSeek/Qwen ChatModel
"""

from __future__ import annotations

import re
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day25_chat_model import BaseMessage, HumanMessage, StarMindChatModel, SystemMessage
from src.day28_loaders import Document
from src.day28_text_splitter import RecursiveCharacterTextSplitter
from src.day29_vector_store import StarMindVectorStore


FALLBACK_ANSWER = (
    "抱歉，星辰知识库中暂未找到与该问题相关的确切依据，无法为您编造答案。"
    "建议联系对应业务部门（HR/财务/IT服务台）人工确认。"
)


@dataclass
class Citation:
    """单条结构化引用溯源对象，供前端展示来源卡片与原文片段。"""

    ref_id: int
    source: str
    section: str
    score: float
    snippet: str


@dataclass
class RAGResponse:
    """RAG 问答统一响应结构体。"""

    question: str
    answer: str
    citations: List[Citation] = field(default_factory=list)
    is_fallback: bool = False
    latency_ms: float = 0.0


def _grounded_rag_handler(messages: List[BaseMessage]) -> str:
    """
    离线确定性 Grounded 生成器（模拟严格遵循 <context> 的企业大模型）：
    从检索到的 [来源N] 片段中提取与问题最相关的事实句子并附带引用编号。
    生产环境替换为 DeepSeek-V3 / Qwen2.5-72B ChatModel 即可。
    """
    user_prompt = messages[-1].content
    # 解析 <context> 中的 [来源N] 块与 <question>
    q_match = re.search(r"<question>(.*?)</question>", user_prompt, re.DOTALL)
    ctx_match = re.search(r"<context>(.*?)</context>", user_prompt, re.DOTALL)
    question = q_match.group(1).strip() if q_match else ""
    context = ctx_match.group(1).strip() if ctx_match else ""

    if not context:
        return FALLBACK_ANSWER

    blocks = re.findall(r"(\[来源\d+\][^\n]*\n.*?)(?=\[来源\d+\]|\Z)", context, re.DOTALL)
    q_chars = set(re.findall(r"[\u4e00-\u9fff]|[a-zA-Z0-9]+", question.lower()))

    selected_lines: List[str] = []
    for block in blocks[:3]:
        header_match = re.match(r"(\[来源\d+\])[^\n]*\n(.*)", block.strip(), re.DOTALL)
        if not header_match:
            continue
        ref_tag, body = header_match.group(1), header_match.group(2).strip()
        # 按句拆分，挑出与提问重合度最高的 1~2 句忠实复述
        sentences = [s.strip() for s in re.split(r"(?<=[。！？；\n])", body) if s.strip()]
        scored = sorted(
            sentences,
            key=lambda s: len(q_chars & set(re.findall(r"[\u4e00-\u9fff]|[a-zA-Z0-9]+", s.lower()))),
            reverse=True,
        )
        best_sent = scored[0] if scored else body[:100]
        selected_lines.append(f"- {best_sent} {ref_tag}")

    if not selected_lines:
        return FALLBACK_ANSWER

    return "根据星辰数科企业知识库记录，为您整理如下：\n" + "\n".join(selected_lines)


class EnterpriseRAGPipeline:
    """
    星辰智脑端到端企业级 RAG 问答流水线：
    支持增量入库、相似度阈值拒答、部门元数据过滤与结构化引用溯源。
    """

    SYSTEM_PROMPT = (
        "你是杭州星辰数科「星辰智脑」企业知识库问答助手。\n"
        "严格遵守以下规则：\n"
        "1. 仅允许基于 <context> 标签内提供的参考资料作答，严禁编造未提及的数字、制度或流程。\n"
        "2. 每个关键事实后必须标注对应的引用编号（如 [来源1]）。\n"
        "3. 如果参考资料不足以回答问题，请直接输出标准拒答提示。"
    )

    def __init__(
        self,
        vector_store: Optional[StarMindVectorStore] = None,
        chunk_size: int = 200,
        chunk_overlap: int = 30,
        top_k: int = 3,
        score_threshold: float = 0.22,
        chat_model: Optional[StarMindChatModel] = None,
    ) -> None:
        self.vector_store = vector_store or StarMindVectorStore(collection_name="starmind_rag")
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )
        self.top_k = top_k
        self.score_threshold = score_threshold
        self.chat_model = chat_model or StarMindChatModel(
            model_name="starmind-rag-llm",
            temperature=0.0,
            handler=_grounded_rag_handler,
        )

    def ingest_documents(self, docs: Sequence[Document]) -> int:
        """离线入库流水线：分块 -> 向量化 -> 写入向量库。"""
        chunks = self.splitter.split_documents(docs)
        ids = self.vector_store.add_documents(chunks)
        return len(ids)

    def ask(
        self,
        question: str,
        where: Optional[Dict[str, Any]] = None,
        top_k: Optional[int] = None,
    ) -> RAGResponse:
        """在线问答流水线：检索 -> 阈值门控 -> 增强生成 -> 溯源封装。"""
        start = time.perf_counter()
        k = top_k or self.top_k
        raw_hits = self.vector_store.similarity_search_with_score(question, k=k, where=where)

        # 第 1 层防幻觉硬护栏：剔除低于相似度阈值的噪音切片
        valid_hits = [(doc, score) for doc, score in raw_hits if score >= self.score_threshold]
        if not valid_hits:
            return RAGResponse(
                question=question,
                answer=FALLBACK_ANSWER,
                citations=[],
                is_fallback=True,
                latency_ms=round((time.perf_counter() - start) * 1000, 2),
            )

        citations: List[Citation] = []
        context_blocks: List[str] = []
        for idx, (doc, score) in enumerate(valid_hits, start=1):
            source = str(doc.metadata.get("source", "未知文档"))
            section = str(doc.metadata.get("section", "正文"))
            citations.append(
                Citation(
                    ref_id=idx,
                    source=source,
                    section=section,
                    score=score,
                    snippet=doc.page_content[:120],
                )
            )
            context_blocks.append(f"[来源{idx}] 文档：{source}（章节：{section}）\n{doc.page_content}")

        context_str = "\n\n".join(context_blocks)
        user_content = f"<context>\n{context_str}\n</context>\n\n<question>{question}</question>"
        ai_msg = self.chat_model.invoke([
            SystemMessage(content=self.SYSTEM_PROMPT),
            HumanMessage(content=user_content),
        ])

        return RAGResponse(
            question=question,
            answer=ai_msg.content,
            citations=citations,
            is_fallback=False,
            latency_ms=round((time.perf_counter() - start) * 1000, 2),
        )


if __name__ == "__main__":
    rag = EnterpriseRAGPipeline()
    n = rag.ingest_documents([
        Document(
            "杭州星辰数科差旅报销标准：一线城市（北上广深）住宿上限600元/晚，新一线城市（杭州等）480元/晚，餐补100元/天。",
            {"source": "财务报销制度_2026.md", "section": "差旅标准", "dept": "财务部"},
        ),
        Document(
            "研发代码发布红线：合并 main 分支需 GitLab MR 至少1位架构师审批，严禁周五17:00后非紧急发布。",
            {"source": "研发发布规范.md", "section": "发布窗口", "dept": "研发部"},
        ),
    ])
    print(f"成功入库切片数: {n}")

    r_hit = rag.ask("去北京出差住酒店报销上限是多少？")
    print("\n【命中场景】\n", r_hit.answer)
    print("引用列表:", [(c.ref_id, c.source, c.score) for c in r_hit.citations])

    r_miss = rag.ask("火星距离地球有多少光年？")
    print("\n【库外拒答场景】\n", r_miss.answer, "| is_fallback =", r_miss.is_fallback)


### 🔍 步骤 1 代码深度解析

1. `EnterpriseRAGPipeline` 完整实现了「硬阈值门控 + 软 Prompt 约束 + 结构化 Citation 对象」，任何问题都有据可查或体面拒答。
2. `RAGResponse` 包含 `latency_ms` 和 `is_fallback` 埋点字段，上线后可直接统计「知识库命中率 / 拒答率 / P95 响应时间」三大核心业务看板指标。

**预期运行输出参考：**
```text
成功入库切片数: 2

【命中场景】
 根据星辰数科企业知识库记录，为您整理如下：
- 杭州星辰数科差旅报销标准：一线城市（北上广深）住宿上限600元/晚，新一线城市（杭州等）480元/晚，餐补100元/天。 [来源1]
引用列表: [(1, '财务报销制度_2026.md', 0.6512)]

【库外拒答场景】
 抱歉，星辰知识库中暂未找到与该问题相关的确切依据，无法为您编造答案。建议联系对应业务部门（HR/财务/IT服务台）人工确认。 | is_fallback = True
```


## 💻 实操步骤 2：步骤二：打造星辰数科企业知识库命令行问答工具（CLI）

- **目标文件**：`versions/day30/src/day30_kb_cli.py`
- **老陈代码旁白**：现在我们把星辰数科真实的四份部门制度文档装进知识库，封装成一个开箱即用的命令行问答工具（支持 `--dept` 部门过滤与美观的引用溯源卡片渲染），方便团队演示和验收！


In [ ]:
"""
Day 30 - 步骤二：星辰数科企业知识库命令行问答工具（CLI）
支持加载企业全量示范制度语料、按部门过滤检索、格式化输出答案与引用溯源卡片。
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import List, Optional

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document, MarkdownLoader
from src.day30_rag_pipeline import EnterpriseRAGPipeline, RAGResponse


STARMIND_CORPUS = [
    (
        "财务报销制度_2026.md",
        "财务部",
        """# 差旅住宿标准
员工因公出差须提前在OA审批。一线城市（北京、上海、深圳、广州）酒店住宿上限每晚600元；新一线城市（杭州、南京、成都）上限每晚480元。
# 餐饮与交通补助
出差期间餐饮补助按每天100元定额随工资发放，无需提供餐饮发票。市内交通实报实销，单次打车超过150元需备注事由。""",
    ),
    (
        "HR员工休假手册_2026.md",
        "HR",
        """# 年假规定
员工在星辰数科连续工作满1年不满10年的，每年享受5天带薪年假；满10年享受10天带薪年假。年假须提前3个工作日提交OA申请。
# 病假与证明
员工申请病假超过1天的，须提供二级及以上公立医院出具的诊断证明与病假条，病假期间按基本工资的80%发放。""",
    ),
    (
        "研发工程规范_v3.md",
        "研发部",
        """# 代码评审与合并
所有业务代码合并至 main 分支前，必须在 GitLab 发起 Merge Request，至少经过1位架构师或高级工程师 Review 通过，且 CI 单元测试通过率须为100%。
# 生产发布窗口
日常生产发布窗口为每周一至周四 10:00-16:00，严禁在周五17:00之后及法定节假日前夕执行非紧急变更发布。""",
    ),
    (
        "IT信息安全指南.md",
        "IT运维",
        """# VPN与内网接入
远程办公访问内网 GitLab 与测试环境必须开启星辰零信任 VPN 客户端，证书每90天自动轮换一次；如遇无法连接，请先在企业微信重置动态令牌。
# 数据脱敏红线
严禁将客户明文手机号、身份证号、银行卡号直接发送给外部公有云大模型 API，所有测试日志须先经脱敏网关处理。""",
    ),
]


def build_starmind_kb(persist_dir: Optional[str | Path] = None) -> EnterpriseRAGPipeline:
    """加载星辰数科内置四部门语料，返回可直接提问的 RAG 引擎。"""
    from src.day29_vector_store import StarMindVectorStore

    store = StarMindVectorStore(collection_name="starmind_enterprise_kb", persist_directory=persist_dir)
    pipeline = EnterpriseRAGPipeline(vector_store=store, chunk_size=180, chunk_overlap=25, score_threshold=0.22)
    if store.count() == 0:
        loader = MarkdownLoader()
        all_docs: List[Document] = []
        for filename, dept, md_text in STARMIND_CORPUS:
            parsed = loader.load_text(md_text, source=filename)
            for d in parsed:
                d.metadata["dept"] = dept
            all_docs.extend(parsed)
        pipeline.ingest_documents(all_docs)
    return pipeline


def format_cli_response(resp: RAGResponse) -> str:
    """将 RAGResponse 格式化为清晰的终端问答卡片。"""
    lines = [
        "=" * 64,
        f"【员工提问】{resp.question}",
        f"【状态】{'兜底拒答 (Fallback)' if resp.is_fallback else '知识库命中'} | 耗时: {resp.latency_ms} ms",
        "-" * 64,
        resp.answer,
    ]
    if resp.citations:
        lines.append("-" * 64)
        lines.append("【引用溯源】")
        for c in resp.citations:
            lines.append(f"  [{c.ref_id}] {c.source} > {c.section} (相似度: {c.score:.3f})")
    lines.append("=" * 64)
    return "\n".join(lines)


def run_demo_queries() -> List[RAGResponse]:
    """运行一组涵盖命中、部门过滤与库外拒答的演示提问。"""
    kb = build_starmind_kb()
    test_cases = [
        ("入职满1年有几天年假？需要提前多久申请？", None),
        ("周五晚上6点可以发版上线吗？", {"dept": "研发部"}),
        ("远程办公连不上内网GitLab怎么办？", None),
        ("请问公司食堂明天中午红烧肉多少钱一份？", None),
    ]
    results: List[RAGResponse] = []
    for q, where in test_cases:
        resp = kb.ask(q, where=where)
        print(format_cli_response(resp))
        results.append(resp)
    return results


if __name__ == "__main__":
    run_demo_queries()


### 🔍 步骤 2 代码深度解析

1. `build_starmind_kb` 会检查 `store.count() == 0` 再执行入库，如果传入了 `persist_dir`，第二次启动 CLI 会秒级复用磁盘索引，避免重复向量化。
2. `format_cli_response` 把状态（命中/拒答）、耗时、正文回答和 `[1] 文档名 > 章节 (相似度)` 清晰分层展示，这正是企业内部机器人卡片的标准信息架构。

**预期运行输出参考：**
```text
================================================================
【员工提问】入职满1年有几天年假？需要提前多久申请？
【状态】知识库命中 | 耗时: 1.12 ms
----------------------------------------------------------------
根据星辰数科企业知识库记录，为您整理如下：
- 员工在星辰数科连续工作满1年不满10年的，每年享受5天带薪年假；满10年享受10天带薪年假。 [来源1]
----------------------------------------------------------------
【引用溯源】
  [1] HR员工休假手册_2026.md > 年假规定 (相似度: 0.642)
================================================================
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day30/tests/test_day30.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 30 自动化单元测试：端到端验证星辰企业知识库 RAG 命中、部门过滤与拒答兜底。"""

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day30_kb_cli import build_starmind_kb, format_cli_response
from src.day30_rag_pipeline import FALLBACK_ANSWER


def test_rag_hit_and_citations(tmp_path):
    kb = build_starmind_kb(persist_dir=tmp_path)
    resp = kb.ask("去上海出差住酒店每晚报销上限是多少？")
    assert resp.is_fallback is False
    assert "600元" in resp.answer
    assert "[来源1]" in resp.answer
    assert len(resp.citations) >= 1
    assert resp.citations[0].source == "财务报销制度_2026.md"


def test_rag_dept_filter(tmp_path):
    kb = build_starmind_kb(persist_dir=tmp_path)
    resp = kb.ask("代码发布窗口规定是什么？", where={"dept": "研发部"})
    assert resp.is_fallback is False
    assert all(c.source == "研发工程规范_v3.md" for c in resp.citations)

    # 用错误的部门过滤应当触发拒答
    resp_wrong_dept = kb.ask("代码发布窗口规定是什么？", where={"dept": "财务部"})
    assert resp_wrong_dept.is_fallback is True


def test_rag_out_of_domain_fallback(tmp_path):
    kb = build_starmind_kb(persist_dir=tmp_path)
    resp = kb.ask("量子纠缠与相对论的数学推导过程是什么？")
    assert resp.is_fallback is True
    assert resp.answer == FALLBACK_ANSWER
    assert resp.citations == []
    card = format_cli_response(resp)
    assert "兜底拒答" in card

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现引用编号一致性校验器（CitationVerifier）

**题目要求：**
大模型有时会「伪造引用」（例如只提供了 2 条参考资料，它却在回答里写了 `[来源5]`）。请编写 `verify_citations(resp: RAGResponse)` 函数，提取回答中所有 `[来源N]` 编号，检查是否全部落在 `1..len(resp.citations)` 合法范围内，返回 `(is_valid, invalid_refs)`。

**思路提示：** 使用正则 `re.findall(r'\[来源(\d+)\]', resp.answer)` 提取引用编号并转为 `int` 校验。


In [ ]:
"""Day 30 作业一参考答案：防伪造引用一致性校验器"""
import re
import sys
from pathlib import Path
from typing import List, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day30_rag_pipeline import Citation, RAGResponse


def verify_citations(resp: RAGResponse) -> Tuple[bool, List[int]]:
    cited_ids = [int(x) for x in re.findall(r"\[来源(\d+)\]", resp.answer)]
    valid_ids = {c.ref_id for c in resp.citations}
    invalid = sorted({cid for cid in cited_ids if cid not in valid_ids})
    return (len(invalid) == 0), invalid


if __name__ == "__main__":
    good = RAGResponse("q", "一线城市上限600元 [来源1]。", [Citation(1, "fin.md", "差旅", 0.8, "...")])
    bad = RAGResponse("q", "一线城市上限600元 [来源1]，且周末双倍补贴 [来源9]。", [Citation(1, "fin.md", "差旅", 0.8, "...")])
    assert verify_citations(good) == (True, [])
    assert verify_citations(bad) == (False, [9])
    print("引用一致性校验通过！伪造引用 [来源9] 已被精准拦截。")


**答案解析与评分标准：**

上线后加一道 10 行代码的引用后校验，能立刻拦截大模型胡乱编造引用编号的幻觉，是金融/法务 RAG 系统的标配。


### 作业 2（企业实战）：作业二：实现基于拒答信号的工单自动转人工路由

**题目要求：**
当 `RAGResponse.is_fallback` 为 True 时，不要只让员工吃闭门羹。请实现 `ask_with_escalation(kb, question, emp_id)`：若命中知识库直接返回答案；若触发拒答，则自动生成一条待人工处理的工单字典（含 `ticket_id`、`emp_id`、`question`、`target_desk` 按关键词分派到 HR/财务/IT/综合服务台）。

**思路提示：** 调用 `kb.ask(question)`，判断 `resp.is_fallback` 后根据关键词路由并构造工单。


In [ ]:
"""Day 30 作业二参考答案：RAG 拒答自动转人工工单闭环"""
import sys
import uuid
from pathlib import Path
from typing import Any, Dict

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day30_kb_cli import build_starmind_kb


def ask_with_escalation(kb, question: str, emp_id: str) -> Dict[str, Any]:
    resp = kb.ask(question)
    if not resp.is_fallback:
        return {"mode": "rag_answered", "answer": resp.answer, "citations": len(resp.citations)}

    if any(k in question for k in ("工资", "薪资", "社保", "离职")):
        desk = "HRBP人工服务台"
    elif any(k in question for k in ("发票", "打款", "税率")):
        desk = "财务共享中心"
    elif any(k in question for k in ("电脑", "账号", "门禁")):
        desk = "IT运维台"
    else:
        desk = "行政综合服务台"

    return {
        "mode": "escalated_ticket",
        "answer": resp.answer,
        "ticket": {
            "ticket_id": f"TK-{uuid.uuid4().hex[:6].upper()}",
            "emp_id": emp_id,
            "question": question,
            "target_desk": desk,
        },
    }


if __name__ == "__main__":
    kb = build_starmind_kb()
    out = ask_with_escalation(kb, "请问公积金缴存比例今年调整了吗？我的社保明细在哪查？", emp_id="E1024")
    print("转人工结果:", out)
    assert out["mode"] == "escalated_ticket" and out["ticket"]["target_desk"] == "HRBP人工服务台"


**答案解析与评分标准：**

优秀的企业 AI 产品从不追求「100% 回答所有问题」，而是追求「懂的答得准、不懂的无缝转人工并反哺知识库补录」。


### 作业 3（架构进阶）：作业三：实现上下文 Token 预算自适应截断组装器

**题目要求：**
当 `top_k=10` 且每块较长时，全塞给大模型可能超出预算。请实现 `pack_context_within_budget(hits, max_chars=260)`：按相似度从高到低依次装入 `[来源N]` 文本块，一旦加入下一块会超过 `max_chars` 即停止（至少保留 Top-1 切片，超长则截断），返回 `(context_str, kept_count)`。

**思路提示：** 遍历排序后的 `(doc, score)` 列表，实时统计已用字符长度。


In [ ]:
"""Day 30 作业三参考答案：预算感知的 RAG 上下文装箱器"""
import sys
from pathlib import Path
from typing import List, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document


def pack_context_within_budget(
    hits: List[Tuple[Document, float]],
    max_chars: int = 200,
) -> Tuple[str, int]:
    blocks: List[str] = []
    used = 0
    for idx, (doc, score) in enumerate(hits, start=1):
        block = f"[来源{idx}] ({doc.metadata.get('source', 'doc')}): {doc.page_content}"
        if not blocks and len(block) > max_chars:
            blocks.append(block[:max_chars])
            break
        if used + len(block) > max_chars:
            break
        blocks.append(block)
        used += len(block)
    return "\n\n".join(blocks), len(blocks)


if __name__ == "__main__":
    sample_hits = [
        (Document("一线城市住宿上限600元，新一线480元，需提前在OA审批通过。" * 2, {"source": "a.md"}), 0.9),
        (Document("餐饮补助按每天100元定额发放，无需提供餐饮发票。" * 2, {"source": "b.md"}), 0.8),
        (Document("市内交通实报实销，单次超过150元需备注说明。" * 2, {"source": "c.md"}), 0.7),
    ]
    ctx, kept = pack_context_within_budget(sample_hits, max_chars=160)
    print(f"保留切片数: {kept}, 总字符数: {len(ctx)}\n{ctx}")
    assert kept < len(sample_hits) and len(ctx) <= 165


**答案解析与评分标准：**

控制上下文预算不仅能省 50% 以上的输入 Token 费用，还能避免「Lost in the Middle（中间迷失）」效应——塞入太多低相关切片反而会干扰大模型注意力。


---

## 🎯 老陈收尾复盘

恭喜各位！今天你们亲手跑通了自己人生中第一套结构完整、带引用溯源和防幻觉拒答机制的企业级 RAG 问答系统！晚上别急着划水，把今天内置的星辰知识库命令行工具多测几个刁钻问题（比如故意问「公司发不发茅台酒」看它会不会乖乖触发拒答），明天周测和消融实验见！
